In [1]:
import cvxpy as cvx
import numpy as np

### Intro to CVXPY
CVXPY is an algebraic modeling language, i.e. a language in which you can formulate your problems and then pass them to a solver. It has a similar purpose but a different feature set to Pyomo. This is a in-class introduction to CVXPY. More documentation can be found at [https://www.cvxpy.org/tutorial/](https://www.cvxpy.org/tutorial/)

Let's start with an easy example:
$$\max_{x_{1}, x_{2}} \quad 2 x_{1} + 3 x_{2}$$
$$\text{s.t.} \quad x_{1} - x_{2} \geq 4$$
$$ 0 \leq x_{1} \leq 8$$
$$0 \leq x_{2} \leq 4$$

In CVXPY, you can declare scalar variables with `cvx.Variable()`.

In [2]:
x1 = cvx.Variable()
x2 = cvx.Variable()

You can declare an objective with `cvx.Minimize` or `cvx.Maximize` as appropriate for your problem

In [3]:
objective = cvx.Maximize(2.*x1 + 3.*x2)

A constraint can be assembled with `<=`, `>=`, or `==` as appropriate. A set of constraints is a python list of constraints.

In [4]:
constraints = [x1 - x2 >= 4.,
               0. <= x1, #compound constraint not allowed, so I break this 1 up
               x1 <= 8.,
               0. <= x2,
               x2 <= 4.]               

Problems in CVXPY are declared with `cvx.Problem`. The first argument is your objective function and the second is your python list of constraints. The second argument is options (in case your problem has no constraints).

In [5]:
problem = cvx.Problem(objective, constraints)

Solve your problem by calling the `solve` method for your `cvx.Problem` variable. You can optionally specify a solver to use, various solver options, and whether you want verbose output.

In [6]:
problem.solve(solver='HIGHS', verbose=True)

(CVXPY) Jul 09 02:14:48 PM: Your problem has 2 variables, 5 constraints, and 0 parameters.
(CVXPY) Jul 09 02:14:48 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Jul 09 02:14:48 PM: DCP verification time: 0.0004 seconds.
(CVXPY) Jul 09 02:14:48 PM: Expression tree has 3 nodes.
(CVXPY) Jul 09 02:14:48 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Jul 09 02:14:48 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Jul 09 02:14:48 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Jul 09 02:14:48 PM: Compiling problem (target solver=HIGHS).
(CVXPY) Jul 09 02:14:48 PM: Reduction chain: FlipObjective -> Dcp2Cone -> CvxAttr2Constr -> EliminateZeroSized -> ConeMatrixStuffing -> HIGHS
(CVXPY) Jul 09 02:14:48 PM: Applying reduction FlipObjective
(CVXPY) Jul 09 02:14:48 PM: Applying reduction Dcp2Cone
(CVXPY) Jul 09 02:14:

                                     CVXPY                                     
                                     v1.9.2                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------


np.float64(28.0)

Access the optimal value of a decision variable using the `value` method of a `cvx.Variable` object.

In [7]:
print('x1 optimal value', x1.value)
print('x2 optimal value', x2.value)
print('optimal objective value', problem.value)

x1 optimal value 8.0
x2 optimal value 4.0
optimal objective value 28.0


CVXPY supports a lot of solvers, of both the open source and proprietary varieties. You can find them all [here](https://www.cvxpy.org/tutorial/solvers/index.html). You can see the solvers you currently have installed with `cvx.installed_solvers`.

In [8]:
print(cvx.installed_solvers())

['CLARABEL', 'SCS', 'GUROBI', 'SCIPY', 'HIGHS', 'OSQP']


### Additional conveniences.

#### Attributes of a Variable

Variables can be declared with certain attributes. An exhaustive list of attributes can be found [here](https://www.cvxpy.org/tutorial/constraints/index.html), though the most important for us are variables which are `nonneg` (non-negative), `nonpos` (non-positive), `integer`, and `boolean`.

In the above example, we can declare `x1` and `x2` as non-negative variables and then omit the non-negativity constraints from the constraint set.

In [9]:
x1 = cvx.Variable(nonneg=True)
x2 = cvx.Variable(nonneg=True)

#### Matrix/Vector Notation

A major advantage of CVXPY over Pyomo is that it can parse optimization problems directly in matrix/vector notation. 

For example, we can write the example optimization problem above as
$$\max_{x \geq 0} \quad \begin{pmatrix} 2 & 3 \end{pmatrix} \begin{pmatrix} x_{1} \\ x_{2} \end{pmatrix}$$
$$\text{s.t.} \quad \begin{pmatrix} -1 & 1 \\ 1 & 0 \\ 0 & 1 \end{pmatrix} \begin{pmatrix} x_{1} \\ x_{2} \end{pmatrix} \leq \begin{pmatrix} -4 \\ 8 \\ 4 \end{pmatrix}.$$


where $x = \begin{pmatrix} x_{1} \\ x_{2} \end{pmatrix}$. This can be parsed directly in CVXPY by defining matrices and vectors appropriately.

In [10]:
x = cvx.Variable(2, nonneg=True) #the first argument is the variable's shape; here a length 2 vector
c = np.array([2,3])
A = np.array([[-1, 1], [1, 0], [0, 1]])
b = np.array([-4, 8, 4])

obj = cvx.Maximize(c.T @ x)
cons = [A @ x <= b]
prob = cvx.Problem(obj, cons)
prob.solve(verbose=True, solver='HIGHS')

(CVXPY) Jul 09 02:14:54 PM: Your problem has 2 variables, 3 constraints, and 0 parameters.
(CVXPY) Jul 09 02:14:54 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Jul 09 02:14:54 PM: DCP verification time: 0.0001 seconds.
(CVXPY) Jul 09 02:14:54 PM: Expression tree has 1 nodes.
(CVXPY) Jul 09 02:14:54 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Jul 09 02:14:54 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Jul 09 02:14:54 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Jul 09 02:14:54 PM: Compiling problem (target solver=HIGHS).
(CVXPY) Jul 09 02:14:54 PM: Reduction chain: FlipObjective -> Dcp2Cone -> CvxAttr2Constr -> EliminateZeroSized -> ConeMatrixStuffing -> HIGHS
(CVXPY) Jul 09 02:14:54 PM: Applying reduction FlipObjective
(CVXPY) Jul 09 02:14:54 PM: Applying reduction Dcp2Cone
(CVXPY) Jul 09 02:14:

                                     CVXPY                                     
                                     v1.9.2                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------


np.float64(28.0)

### Variable Bounds
We can also declare the variables with bounds instead of enforcing them in the constraints. In that case we get the following.

In [11]:
x = cvx.Variable(2, bounds=(0.,(8,4))) #bounds can be constant for all entries in vectors, or different for all entries, or a mix of both
c = np.array([2,3])
A = np.array([[-1, 1]])
b = np.array([-4])

obj = cvx.Maximize(c.T @ x)
cons = [A @ x <= b]
prob = cvx.Problem(obj, cons)
prob.solve(verbose=True, solver='HIGHS')

(CVXPY) Jul 09 02:15:11 PM: Your problem has 2 variables, 1 constraints, and 0 parameters.
(CVXPY) Jul 09 02:15:11 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Jul 09 02:15:11 PM: DCP verification time: 0.0001 seconds.
(CVXPY) Jul 09 02:15:11 PM: Expression tree has 1 nodes.
(CVXPY) Jul 09 02:15:11 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Jul 09 02:15:11 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Jul 09 02:15:11 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Jul 09 02:15:11 PM: Compiling problem (target solver=HIGHS).
(CVXPY) Jul 09 02:15:11 PM: Reduction chain: FlipObjective -> Dcp2Cone -> CvxAttr2Constr -> EliminateZeroSized -> ConeMatrixStuffing -> HIGHS
(CVXPY) Jul 09 02:15:11 PM: Applying reduction FlipObjective
(CVXPY) Jul 09 02:15:11 PM: Applying reduction Dcp2Cone
(CVXPY) Jul 09 02:15:

                                     CVXPY                                     
                                     v1.9.2                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------


np.float64(28.0)

#### Sparse Matrices
The final feature we will disucss today is CVXPY's ability to work with *sparse* matrices. A sparse matrix (mathematically) is a matrix with many zeros. Computationally, sparse matrices are stored in a sparse matrix format, which saves memory by not storing all of the zero values. The `scipy.sparse` module implements a handful of sparse matrix formats. They are:
- COO (COOdinate format)
- CSC (Compressed Sparse Column format)
- CSR (Compressed Sparse Row format)
- DOK (Dictionary of Keys format)
- LIL (LIst of Lists format)
- BSR (Block Sparse Row)

More documentation [here](https://docs.scipy.org/doc/scipy/reference/sparse.html). These formats are good at different tasks. In general, to take advantage of their capabilities, you can/should use DOK or COO to build sparse matrices, then convert to either CSC or CSR to compute with them.

Here's an example of how you can use a sparse matrix for the example problem we've solved in this notebook (though it is completely unecessary because the problem is small).

In [15]:
import scipy.sparse as sparse

In [13]:
A_sp = sparse.coo_array((3,2)) #empty 3 x 2 array
print(A_sp)

<COOrdinate sparse array of dtype 'float64'
	with 0 stored elements and shape (3, 2)>


In [16]:
A_sp = sparse.dok_array((3,2)) #empty 3 x 2 array
print(A_sp)

<Dictionary Of Keys sparse array of dtype 'float64'
	with 0 stored elements and shape (3, 2)>


In [20]:
import scipy
scipy.__version__

'1.16.2'

In [22]:
pip install --upgrade scipy==1.18.0

   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   ---------------------------------------- 0.3/36.6 MB ? eta -:--:--
   ---------------------------------------- 0.3/36.6 MB ? eta -:--:--
    --------------------------------------- 0.5/36.6 MB 504.9 kB/s eta 0:01:12
    --------------------------------------- 0.5/36.6 MB 504.9 kB/s eta 0:01:12
    --------------------------------------- 0.8/36.6 MB 591.0 kB/s eta 0:01:01
    --------------------------------------- 0.8/36.6 MB 591.0 kB/s eta 0:01:01
    --------------------------------------- 0.8/36.6 MB 591.0 kB/s eta 0:01:01
   - -------------------------------------- 1.0/36.6 MB 547.6 kB/s eta 0:01:05
   - -------------------------------------- 1.3/36.6 MB 604.3 kB/s eta 0:00:59
   - -------------------------------------- 1.3/36.6 MB 604.3 kB/s eta 0:00:59
   - -------------

In [17]:
#but now we fill it in with elts
A_sp[0,0] = -1.
A_sp[0,1] = 1.
A_sp[1,0] = 1.
A_sp[2,1] = 1.

In [23]:
#Finally, we can use the sparse matrix A_sp to define
#our problem in cvxpy instead of the dense matrix A
obj = cvx.Maximize(c.T @ x)
cons = [A_sp @ x <= b]
prob = cvx.Problem(obj, cons)
prob.solve(verbose=True, solver='HIGHS')

(CVXPY) Jul 09 02:34:00 PM: Your problem has 2 variables, 3 constraints, and 0 parameters.
(CVXPY) Jul 09 02:34:00 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Jul 09 02:34:00 PM: DCP verification time: 0.0002 seconds.
(CVXPY) Jul 09 02:34:00 PM: Expression tree has 1 nodes.
(CVXPY) Jul 09 02:34:00 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Jul 09 02:34:00 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Jul 09 02:34:00 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Jul 09 02:34:00 PM: Compiling problem (target solver=HIGHS).
(CVXPY) Jul 09 02:34:00 PM: Reduction chain: FlipObjective -> Dcp2Cone -> CvxAttr2Constr -> EliminateZeroSized -> ConeMatrixStuffing -> HIGHS
(CVXPY) Jul 09 02:34:00 PM: Applying reduction FlipObjective
(CVXPY) Jul 09 02:34:00 PM: Applying reduction Dcp2Cone
(CVXPY) Jul 09 02:34:

                                     CVXPY                                     
                                     v1.9.2                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------


-inf

**Parting Thought**: All algebraic modeling languages (i.e. Pyomo, CVXPY) are essentially just tools for passing a sparse matrix, which the solver requires for its computation, to a solver in a user-friendly manner.